# NDVI Field Analysis and Web Map Visualization

This notebook demonstrates a simple geospatial workflow:

1. Load agricultural field polygons and OSM road data  
2. Calculate mean NDVI values for each field  
3. Reproject raster data for web visualization  
4. Create an interactive map using Folium

In [1]:
# ==========================================
# 1 Imports
# ==========================================

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import array_bounds
from shapely.geometry import mapping

import folium
import matplotlib.pyplot as plt

In [2]:
# -------------------------
# 2 Daten laden / Load data
# -------------------------

from pathlib import Path




PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

base_path = PROJECT_ROOT / "data" / "processed" / "examples"
osm_path = base_path / "ways_osm.geojson"
agrar_path = base_path / "agrar_test.geojson"
test_area_path = base_path / "test_area.geojson"
ndvi_path = base_path / "ndvi_clipped.TIF"

osm_ways = gpd.read_file(osm_path)
agrar = gpd.read_file(agrar_path)
test_area = gpd.read_file(test_area_path)

In [3]:
# 3. Datumsfelder fixen / Fix date column

def fix_datetime_columns(gdf):
    for col in gdf.columns:
        if pd.api.types.is_datetime64_any_dtype(gdf[col]):
            gdf[col] = gdf[col].astype(str)
    return gdf

osm_ways = fix_datetime_columns(osm_ways)
agrar = fix_datetime_columns(agrar)
test_area = fix_datetime_columns(test_area)

In [4]:
# 4. CRS für Webmap / Set CRS for webmap

agrar = agrar.to_crs("EPSG:4326")
test_area = test_area.to_crs("EPSG:4326")

In [6]:
# 5. NDVI pro Agrarfläche berechnen / Calculate NDVI per agricultural area

mean_ndvi_list = []

with rasterio.open(ndvi_path) as src:

    agrar_raster = agrar.to_crs(src.crs)

    for geom in agrar_raster.geometry:

        out_image, out_transform = mask(src, [mapping(geom)], crop=True)

        ndvi_vals = out_image[0]
        ndvi_vals = ndvi_vals[ndvi_vals != src.nodata]

        if len(ndvi_vals) > 0:
            mean_ndvi_list.append(ndvi_vals.mean())
        else:
            mean_ndvi_list.append(np.nan)

agrar["mean_ndvi"] = mean_ndvi_list

In [ ]:
# 6. NDVI Raster für Webkarte reprojizieren / Reproject NDVI raster for web map

with rasterio.open(ndvi_path) as src:

    dst_crs = "EPSG:4326"

    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds
    )

    ndvi_4326 = np.empty((1, height, width), dtype=np.float32)

    reproject(
        source=src.read(1),
        destination=ndvi_4326[0],
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
        src_nodata=src.nodata,
        dst_nodata=np.nan
    )

In [ ]:
# 7. NDVI einfärben / NDVI staining

ndvi_norm = (
    (ndvi_4326[0] - np.nanmin(ndvi_4326[0])) /
    (np.nanmax(ndvi_4326[0]) - np.nanmin(ndvi_4326[0]))
)

cmap = plt.cm.RdYlGn
ndvi_colored = cmap(ndvi_norm)

ndvi_img = (ndvi_colored[:, :, :3] * 255).astype(np.uint8)

In [ ]:
# 8. Raster-Bounds berechnen / Calculate Raster-Bounds

ndvi_bounds = array_bounds(height, width, transform)

bounds_folium = [
    [ndvi_bounds[1], ndvi_bounds[0]],
    [ndvi_bounds[3], ndvi_bounds[2]]
]

In [ ]:
# 9. Karte erstellen / Create map

center = [
    test_area.geometry.centroid.y.mean(),
    test_area.geometry.centroid.x.mean()
]

m = folium.Map(location=center, zoom_start=13)


C:\Users\frank\AppData\Local\Temp\ipykernel_15664\3492808762.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  test_area.geometry.centroid.y.mean(),
C:\Users\frank\AppData\Local\Temp\ipykernel_15664\3492808762.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  test_area.geometry.centroid.x.mean()


In [ ]:
# 10. NDVI Overlay hinzufügen / add NDVI overlay

folium.raster_layers.ImageOverlay(
    image=ndvi_img,
    bounds=bounds_folium,
    opacity=0.6,
    name="NDVI"
).add_to(m)

In [ ]:
# 11. Vektordaten hinzufügen / add vector data to map

# OSM Wege / ways

folium.GeoJson(
    osm_ways,
    name="OSM Wege",
    style_function=lambda x: {"color": "blue", "weight": 2}
).add_to(m)

# Agrarflächen / Agricultural land

folium.GeoJson(
    agrar,
    name="Agrarflächen",
    style_function=lambda x: {
        "color": "green",
        "weight": 1,
        "fillOpacity": 0.3
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["mean_ndvi"],
        aliases=["Mittlerer NDVI"]
    )
).add_to(m)

# Untersuchungs- bzw. Testgebiet / Testarea

folium.GeoJson(
    test_area,
    name="Testgebiet",
    style_function=lambda x: {
        "color": "red",
        "weight": 2,
        "fillOpacity": 0
    }
).add_to(m)





In [ ]:
# 12. Layer Control

folium.LayerControl().add_to(m)

# 14. Karte anzeigen / Show map

# m



In [ ]:
# Karte speichern / save map 

m.save("map.html")

In [ ]:
# Karte im Browser anzeigen lassen / Show map in browser
import webbrowser
webbrowser.open("map.html")

True